# Hyperparameter tuning of RSGM

A search over the hyperparameters of RSGM (`hp/models/rsgm.py`) with [Optuna](https://optuna.org), on every 30th frame of AMASS (`~/data/AMASS_rotations_c30`, official split), in two rounds. The defaults (`config/models/rsgm.yaml`) are the settings of the SO(3) experiment of riemannian-score-sde with β_f = 10. The setup, the results and what they mean are written up in `docs/rsgm_hyperparameter_tuning.md`.

**The objective** is the mean log-likelihood of a fixed, stratified sample of validation poses (`N_PER_DATASET` of each c30 valid dataset; EMA weights; `OBJECTIVE_STEPS` ODE steps), after `TRIAL_ITERATIONS` training steps. Not the validation loss: its value depends on the hyperparameters themselves (noise schedule, `loss_t_min`, weighting), so it does not compare between configurations. The test split is not used here.

**The search.** TPE sampler; a median pruner checks the log-likelihood (half of the validation poses, `PRUNE_STEPS` ODE steps) every `PRUNE_EVERY` steps. The studies live in SQLite (`STUDY_DIR/optuna.db`): the notebook can be stopped and run again, it continues where it stopped, and `optuna-dashboard sqlite:///$HOME/data/experiments/rsgm_hpo/optuna.db` shows them live.
- **Round 1** (`rsgm_c30`, 40 trials): a broad space of ten parameters; the default configuration is the first trial.
- **Round 2** (`rsgm_c30_round2`, 10 new trials): five parameters around the best of round 1, the rest fixed; warm-started with the round-1 trials that lie in the new space.
- **Final retraining** (optional, `RUN_FINAL`): the best configurations at the full budget with several seeds, the choice on those runs.

In [ ]:
import json
import logging
import os
import time
from pathlib import Path
from unittest import mock

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import torch
from omegaconf import OmegaConf

from hp.data.amass import AMASS
from hp.data.amass_cfg import ORIGINAL_AMASS_SPLITS, AmassJoints4Learning
from hp.models import build_model
from hp.models.io import load_model, save_model
from hp.training import TrainConfig, Trainer, cli

# ---- parameters ----------------------------------------------------------------------
ROTATIONS = "~/data/AMASS_rotations_c30"   # training and validation data: every 30th frame
SPLIT = "original"                         # the official AMASS split
STUDY_DIR = "~/data/experiments/rsgm_hpo"  # the Optuna database and the weights of the trials
DEVICE = "mps"                             # "cpu", "mps" or "cuda:0"
SEED = 10
TRIAL_ITERATIONS = 30_000                  # training steps of a trial (the same in both rounds)
PRUNE_EVERY = 10_000                       # steps between the checks of the pruner
N_PER_DATASET = 500                        # validation poses per valid dataset (4 datasets)
OBJECTIVE_STEPS = 200                      # ODE steps of the objective's log-likelihood
PRUNE_STEPS = 100                          # ODE steps of the pruner's checks (on half of the poses)
WARMUP_STEPS = 500                         # linear warm-up of the cosine schedule

STUDY_NAME_1, N_TRIALS_1 = "rsgm_c30", 40
STUDY_NAME_2, N_TRIALS_2 = "rsgm_c30_round2", 10   # round 2: new trials, the warm start comes on top

RUN_FINAL = False                          # True: retrain the best configurations at the full budget (hours)
TOP_K = 3                                  # best configurations of round 2 retrained (plus the default)
FINAL_ROOT = "~/data/experiments/rsgm_tuned"  # their runs: FINAL_ROOT/<config>/run_0001, ...
FINAL_ITERATIONS = 100_000
FINAL_SEEDS = [10, 11, 12]
# ----------------------------------------------------------------------------------------

expand = lambda p: Path(p).expanduser()
STUDY_DIR, FINAL_ROOT = expand(STUDY_DIR), expand(FINAL_ROOT)
STUDY_DIR.mkdir(parents=True, exist_ok=True)
STORAGE = f"sqlite:///{STUDY_DIR / 'optuna.db'}"
States = optuna.trial.TrialState
optuna.logging.set_verbosity(optuna.logging.WARNING)
logging.getLogger().setLevel(logging.WARNING)

# the defaults of config/models/rsgm.yaml, in the terms of the search space
DEFAULT = dict(lr=2e-4, batch_size=512, width=512, depth=5, activation="sin", beta_f=10.0, loss_t_min=1e-3,
               ema_decay=0.999, scheduler="step", weighting="std")

## 1. Data

The training split (c30) is loaded once and shared by all trials. The validation poses: `N_PER_DATASET` random poses of every valid dataset (HumanEva, HDM05, SFU, MoSh), drawn with a fixed seed. Without the stratification HDM05 (79% of the split) would dominate. The first half of each dataset's poses is the subset of the pruner's checks.

In [ ]:
with mock.patch.dict(os.environ, {"HUPROSO_AMASS_SPLIT": SPLIT}):
    train_set = AMASS(mode="train", path_to_dataset=ROTATIONS, rotation_representation="matrix",
                      selected_joints=AmassJoints4Learning, load_dtype="float32")

rng = np.random.default_rng(SEED)
parts, half = [], []
for name in ORIGINAL_AMASS_SPLITS["valid"]:
    poses = AMASS(datasets=[name], path_to_dataset=ROTATIONS, rotation_representation="matrix",
                  selected_joints=AmassJoints4Learning).data.reshape(-1, len(AmassJoints4Learning), 3, 3)
    chosen = poses[rng.choice(len(poses), min(N_PER_DATASET, len(poses)), replace=False)]
    parts.append(chosen)
    half.append(chosen[: len(chosen) // 2])
    print(f"valid {name:9s}: {len(poses):6,} poses, {len(chosen)} chosen")
valid_poses, prune_poses = torch.cat(parts), torch.cat(half)
print(f"train: {len(train_set):,} poses; validation poses: {len(valid_poses):,} (objective), {len(prune_poses):,} (pruner)")


@torch.no_grad()
def log_likelihood(model, poses, steps, batch=250):
    # Log-likelihoods [N] of the poses under the model, with `steps` ODE steps.
    old, model.likelihood_steps = model.likelihood_steps, steps
    try:
        return torch.cat([model.log_prob(poses[i:i + batch].to(model.device)).float().cpu() for i in range(0, len(poses), batch)])
    finally:
        model.likelihood_steps = old

## 2. Search spaces and the objective

`suggest_round1` and `suggest_round2` turn a trial into the arguments of the model and the fields of `TrainConfig`; `split_params` does the same for a plain dict of parameters (the default configuration, or a trial's parameters plus the fixed ones of round 2).

A trial builds the model, trains it with `Trainer`, reports the pruner's log-likelihood every `PRUNE_EVERY` steps, and returns the mean log-likelihood of the validation poses (EMA weights). A trial whose loss or likelihood stops being finite is pruned. The EMA weights of a finished trial are saved under `STUDY_DIR`.

In [ ]:
def split_params(p):
    # A dict of search parameters -> (arguments of the model, fields of TrainConfig).
    model_args = dict(hidden=[p["width"]] * p["depth"], activation=p["activation"], beta_f=p["beta_f"],
                      loss_t_min=p["loss_t_min"], weighting=p["weighting"], likelihood_steps=OBJECTIVE_STEPS)
    train = dict(lr=p["lr"], batch_size=p["batch_size"], ema_decay=p["ema_decay"], scheduler=p["scheduler"],
                 warmup_steps=WARMUP_STEPS if p["scheduler"] == "cosine" else 0)
    return model_args, train


def suggest_round1(trial):
    # Round 1: a broad space of ten parameters.
    return split_params(dict(
        lr=trial.suggest_float("lr", 5e-5, 2e-3, log=True),
        batch_size=trial.suggest_categorical("batch_size", [256, 512, 1024]),
        width=trial.suggest_categorical("width", [256, 512, 1024]),
        depth=trial.suggest_int("depth", 3, 6),
        activation=trial.suggest_categorical("activation", ["sin", "silu"]),
        beta_f=trial.suggest_float("beta_f", 6.0, 16.0),
        loss_t_min=trial.suggest_float("loss_t_min", 1e-4, 1e-2, log=True),
        ema_decay=trial.suggest_categorical("ema_decay", [0.999, 0.9995, 0.9999]),
        scheduler=trial.suggest_categorical("scheduler", ["step", "cosine"]),
        weighting=trial.suggest_categorical("weighting", ["std", "likelihood"]),
    ))


# Round 2: five parameters around the best of round 1, the rest fixed at its winners.
FIXED_2 = dict(activation="silu", batch_size=512, ema_decay=0.999, scheduler="step", weighting="std")
D = optuna.distributions
SPACE_2 = {  # exactly the distributions suggest_round2 creates, so that round-1 trials can be added to the study
    "width": D.CategoricalDistribution([1024, 1536, 2048]),
    "depth": D.IntDistribution(3, 5),
    "lr": D.FloatDistribution(6e-4, 4e-3, log=True),
    "beta_f": D.FloatDistribution(4.0, 10.0),
    "loss_t_min": D.FloatDistribution(1e-3, 1e-2, log=True),
}


def suggest_round2(trial):
    s = SPACE_2
    return split_params(dict(
        width=trial.suggest_categorical("width", s["width"].choices),
        depth=trial.suggest_int("depth", s["depth"].low, s["depth"].high),
        lr=trial.suggest_float("lr", s["lr"].low, s["lr"].high, log=True),
        beta_f=trial.suggest_float("beta_f", s["beta_f"].low, s["beta_f"].high),
        loss_t_min=trial.suggest_float("loss_t_min", s["loss_t_min"].low, s["loss_t_min"].high, log=True),
        **FIXED_2,
    ))


def in_space_2(params):
    # Whether a round-1 configuration lies in the space of round 2.
    if any(params.get(k) != v for k, v in FIXED_2.items()):
        return False
    for name, dist in SPACE_2.items():
        value = params[name]
        inside = value in dist.choices if isinstance(dist, D.CategoricalDistribution) else dist.low <= value <= dist.high
        if not inside:
            return False
    return True


def objective(trial, suggest_fn, weights_dir):
    model_args, train = suggest_fn(trial)
    torch.manual_seed(SEED + trial.number)
    model = build_model("rsgm", num_joints=len(AmassJoints4Learning), **model_args)
    config = TrainConfig(max_iterations=TRIAL_ITERATIONS, valid_interval=PRUNE_EVERY, checkpoint_every=0,
                         device=DEVICE, progress=False, **train)
    trainer = Trainer(model, train_set, None, config)

    def check(step, _):
        if step >= TRIAL_ITERATIONS:
            return  # the end: the objective below
        if not np.isfinite(trainer.history.loss[-1]):
            raise optuna.TrialPruned(f"non-finite loss at step {step}")
        ll = log_likelihood(trainer.ema.model, prune_poses, PRUNE_STEPS)
        value = float(ll.mean()) if torch.isfinite(ll).all() else -np.inf
        trial.report(value, step)
        print(f"  trial {trial.number:3d} step {step:6,}: log-likelihood {value:8.2f}", flush=True)
        if not np.isfinite(value) or trial.should_prune():
            raise optuna.TrialPruned(f"log-likelihood {value:.2f} at step {step}")

    start = time.perf_counter()
    try:
        trainer.fit(callback=check)
        ema = trainer.ema.model
        ll = log_likelihood(ema, valid_poses, OBJECTIVE_STEPS)
        if not torch.isfinite(ll).all():
            raise optuna.TrialPruned("non-finite log-likelihood")
        trial.set_user_attr("parameters", model.num_parameters())
        trial.set_user_attr("sem", float(ll.std() / np.sqrt(len(ll))))
        save_model(ema, Path(weights_dir) / f"trial_{trial.number:04d}")
        return float(ll.mean())
    finally:
        trial.set_user_attr("seconds", time.perf_counter() - start)
        del trainer, model
        if DEVICE == "mps":
            torch.mps.empty_cache()


def report(study, trial):
    # Optuna callback: one line per finished trial.
    value = f"{trial.value:.3f}" if trial.value is not None else "-"
    best = "-"
    if any(t.state == States.COMPLETE for t in study.trials):
        b = study.best_trial
        best = f"{study.best_value:.3f} (trial {b.number}" + (f", round-1 trial {b.user_attrs['round1_trial']})" if "round1_trial" in b.user_attrs else ")")
    print(f"trial {trial.number:3d} {trial.state.name:8s} {trial.user_attrs.get('seconds', 0) / 60:5.1f} min, "
          f"log-likelihood {value}, best {best}", flush=True)


def run_study(study, n_trials, suggest_fn, weights_dir, count=lambda t: True):
    # Run the trials still missing to reach n_trials finished (complete or pruned) ones among those `count` selects.
    done = [t for t in study.trials if count(t) and t.state in (States.COMPLETE, States.PRUNED)]
    missing = max(0, n_trials - len(done))
    print(f"{study.study_name}: {len(done)} trials done, {missing} to run")
    start = time.perf_counter()
    study.optimize(lambda trial: objective(trial, suggest_fn, weights_dir), n_trials=missing, callbacks=[report],
                   catch=(RuntimeError,))
    print(f"{study.study_name}: {(time.perf_counter() - start) / 3600:.1f} h in this run")


def trials_table(study, extra=()):
    # All trials of a study, best first.
    rows = [{"number": t.number, "state": t.state.name, "value": t.value, **t.params, **{k: t.user_attrs.get(k) for k in extra},
             "params (M)": (t.user_attrs.get("parameters") or float("nan")) / 1e6, "minutes": t.user_attrs.get("seconds", float("nan")) / 60}
            for t in study.trials]
    return pd.DataFrame(rows).sort_values("value", ascending=False)


pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 30)

## 3. Round 1: a broad search

| parameter | range | default | why |
|---|---|---|---|
| `lr` | 5e-5 ... 2e-3 (log) | 2e-4 | the main lever |
| `batch_size` | 256, 512, 1024 | 512 | |
| `width`, `depth` of the MLP | 256 / 512 / 1024, 3 ... 6 | 512, 5 | capacity |
| `activation` | sin, silu | sin | sin as in the paper, silu is the usual choice |
| `beta_f` | 6 ... 16 | 10 | speed of the noising; small values leave the end non-uniform |
| `loss_t_min` | 1e-4 ... 1e-2 (log) | 1e-3 | the smallest noise level trained, the hardest regime |
| `ema_decay` | 0.999, 0.9995, 0.9999 | 0.999 | |
| `scheduler` | step, cosine (with warm-up) | step | cosine as in the JAX code |
| `weighting` | std, likelihood | std | likelihood: the loss weighted by β(t), a bound on the negative log-likelihood (Song et al. 2021) |

Fixed: `eps` = 2e-4 (it defines what log p means), β_0, the input (rotation matrices and t), gradient clipping 1.0. The first 10 trials are random (TPE's start-up), the first of them the default configuration.

In [ ]:
study1 = optuna.create_study(
    study_name=STUDY_NAME_1, storage=STORAGE, load_if_exists=True, direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED, multivariate=True, n_startup_trials=10),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=8, n_warmup_steps=0),
)
if not study1.trials:
    study1.enqueue_trial(DEFAULT)
run_study(study1, N_TRIALS_1, suggest_round1, STUDY_DIR / "trials")

In [ ]:
df1 = trials_table(study1, extra=("sem",))
print(df1.head(15).to_string(index=False, float_format=lambda x: f"{x:.4g}"))
print(f"\nstates: {df1.state.value_counts().to_dict()}; the default configuration (trial 0): {study1.trials[0].value:.2f}")

complete1 = df1[df1.state == "COMPLETE"]
print("\nmean / max objective by value of the categorical parameters:")
for p in ["width", "depth", "activation", "batch_size", "ema_decay", "scheduler", "weighting"]:
    stats = complete1.groupby(p)["value"].agg(["count", "mean", "max"]).round(2)
    print(f"  {p:10s} " + "; ".join(f"{k}: n={r['count']:.0f} mean {r['mean']:.1f} max {r['max']:.1f}" for k, r in stats.iterrows()))

In [ ]:
complete = [t for t in study1.trials if t.state == States.COMPLETE]
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot([t.number for t in complete], [t.value for t in complete], "o", label="trial")
ax.plot([t.number for t in complete], np.maximum.accumulate([t.value for t in complete]), "-", label="best so far")
ax.axhline(study1.trials[0].value, color="0.5", ls="--", label="default configuration")
ax.set_xlabel("trial")
ax.set_ylabel("validation log-likelihood")
ax.set_title("round 1")
ax.legend()
plt.show()

# fANOVA importances: unreliable with ~35 correlated trials (in the random start-up, small widths came with small beta_f,
# and the method blames beta_f for what the width did) -- read them next to the grouped statistics above.
importances = optuna.importance.get_param_importances(study1)
print("fANOVA importances:", {k: round(v, 3) for k, v in importances.items()})
optuna.visualization.matplotlib.plot_slice(study1, params=["width", "lr", "beta_f", "depth", "loss_t_min", "ema_decay"])
plt.show()

## 4. Round 2: a narrower search around the best of round 1

What round 1 showed: the ten best trials all have **width 1024** (the upper end of the range), silu, batch 512, EMA 0.999, the step schedule and the std weighting; their **lr** is 8e-4 ... 1.8e-3, rising with the objective (correlation +0.73 at width 1024); their **β_f** is 6.8 ... 9.0, falling with it (−0.74); depth 4 is the best on average, depth 6 bad; `loss_t_min` 1.2e-3 ... 5.5e-3. So round 2 searches only where it matters, and moves the bounds that the best trials pressed against:

| parameter | round 1 | round 2 |
|---|---|---|
| `width` | 256 / 512 / 1024 | 1024 / 1536 / 2048 |
| `depth` | 3 ... 6 | 3 ... 5 |
| `lr` | 5e-5 ... 2e-3 | 6e-4 ... 4e-3 (log) |
| `beta_f` | 6 ... 16 | 4 ... 10 |
| `loss_t_min` | 1e-4 ... 1e-2 | 1e-3 ... 1e-2 (log) |
| activation, batch, EMA, scheduler, weighting | searched | fixed: silu, 512, 0.999, step, std |

A small β_f leaves the end of the noising non-uniform, which biases the likelihood *down* (about −0.2 nats at 6), so a trend towards small β_f is real despite it; below ~4 sampling from a uniform start would suffer, so the range stops there.

- **A new study** in the same database (changing the space of a study confuses TPE).
- **Warm start:** the round-1 trials that lie in the new space are added with their values (no retraining), so TPE models the space from the start.
- **The same protocol** as round 1, else the added values would not compare. Weights of the new trials go to `STUDY_DIR/trials_round2`.

In [ ]:
study2 = optuna.create_study(
    study_name=STUDY_NAME_2, storage=STORAGE, load_if_exists=True, direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED + 1, multivariate=True, n_startup_trials=10),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=8, n_warmup_steps=0),
)
if not study2.trials:  # the warm start, once
    warm = [t for t in study1.trials if t.state == States.COMPLETE and in_space_2(t.params)]
    for t in warm:
        study2.add_trial(optuna.trial.create_trial(
            params={k: t.params[k] for k in SPACE_2}, distributions=SPACE_2, value=t.value,
            intermediate_values=t.intermediate_values, user_attrs={**t.user_attrs, "round1_trial": t.number}))
    print(f"warm start: {len(warm)} round-1 trials in the new space: {[t.number for t in warm]}")
run_study(study2, N_TRIALS_2, suggest_round2, STUDY_DIR / "trials_round2", count=lambda t: "round1_trial" not in t.user_attrs)

In [ ]:
df2 = trials_table(study2, extra=("round1_trial",))
print(df2.to_string(index=False, float_format=lambda x: f"{x:.4g}"))
new = df2[df2.round1_trial.isna()]
print(f"\nnew trials: {new.state.value_counts().to_dict()}; best new {new.value.max():.2f}, "
      f"best added from round 1 {df2[df2.round1_trial.notna()].value.max():.2f}")
print("widths proposed by TPE in round 2:", sorted(new.width.unique().tolist()))

complete2 = [t for t in study2.trials if t.state == States.COMPLETE]
fig, axes = plt.subplots(1, len(SPACE_2), figsize=(4 * len(SPACE_2), 3.4), sharey=True)
for ax, name in zip(axes, SPACE_2):
    for label, color in (("round 1", "0.6"), ("round 2", "C1")):
        ts = [t for t in complete2 if ("round1_trial" in t.user_attrs) == (label == "round 1")]
        ax.scatter([t.params[name] for t in ts], [t.value for t in ts], color=color, label=label)
    if name in ("lr", "loss_t_min"):
        ax.set_xscale("log")
    ax.set_xlabel(name)
axes[0].set_ylabel("validation log-likelihood")
axes[0].legend()
axes[0].set_ylim(np.percentile([t.value for t in complete2], 10), max(t.value for t in complete2) + 0.5)
fig.suptitle("round 2: the objective against each searched parameter")
fig.tight_layout()
plt.show()

## 5. The best configuration

The best trial of round 2 (the added round-1 trials compete too), as overrides for `rsgm_train.ipynb`.

In [ ]:
def as_overrides(params):
    model_args, train = split_params(params)
    return [f"model.args.hidden={json.dumps(model_args['hidden'])}", f"model.args.activation={model_args['activation']}",
            f"model.args.beta_f={model_args['beta_f']}", f"model.args.loss_t_min={model_args['loss_t_min']}",
            f"model.args.weighting={model_args['weighting']}", f"train.lr={train['lr']}", f"train.batch_size={train['batch_size']}",
            f"train.ema_decay={train['ema_decay']}", f"train.scheduler={train['scheduler']}"]


best2 = study2.best_trial
origin = f"round-1 trial {best2.user_attrs['round1_trial']}" if "round1_trial" in best2.user_attrs else f"round-2 trial {best2.number}"
print(f"best: {origin}, validation log-likelihood {best2.value:.2f} after {TRIAL_ITERATIONS:,} steps "
      f"(the default configuration: {study1.trials[0].value:.2f})")
print("for rsgm_train.ipynb:\nTUNED = [" + ",\n         ".join(json.dumps(o) for o in as_overrides({**best2.params, **FIXED_2})) + "]")

## 6. Final retraining (optional)

With `RUN_FINAL = True`: the `TOP_K` best finished trials of round 2 and the default configuration, each trained `FINAL_ITERATIONS` steps with every seed of `FINAL_SEEDS` through `hp.training.cli` (ordinary runs of the model zoo in `FINAL_ROOT/<config>/run_<n>`, which `rsgm_huproso_comparison.ipynb` can evaluate), then evaluated on the same validation poses. The spread over the seeds is the noise a difference between configurations has to beat; the 30k-step differences of the search shrink at the full budget. Finished runs are not trained again. This takes hours.

In [ ]:
def overrides(params, seed, label):
    return ["model=rsgm", f"run_root={FINAL_ROOT}", f"run_name={label}", f"seed={seed}", f"data.rotations={ROTATIONS}",
            f"data.split={SPLIT}", "data.load_dtype=float32", f"train.device={DEVICE}", f"train.max_iterations={FINAL_ITERATIONS}",
            "train.valid_interval=5000", "train.checkpoint_every=0", "train.progress=false", "tensorboard=false",
            *as_overrides(params)]


def finished_runs(label):
    # seed -> run folder of the finished runs of a configuration
    runs = {}
    for run in sorted((FINAL_ROOT / label).glob("run_[0-9][0-9][0-9][0-9]")):
        if (run / "history.json").is_file():
            runs[int(OmegaConf.load(run / "config.yaml").seed)] = run
    return runs


best_trials = sorted(complete2, key=lambda t: t.value, reverse=True)[:TOP_K]
finalists = {"default": DEFAULT, **{f"round2_trial_{t.number:04d}": {**t.params, **FIXED_2} for t in best_trials}}
if RUN_FINAL:
    for label, params in finalists.items():
        for seed in FINAL_SEEDS:
            if seed in finished_runs(label):
                print(f"{label} seed {seed}: already trained in {finished_runs(label)[seed]}")
                continue
            print(f"{label} seed {seed}: training {FINAL_ITERATIONS:,} steps ...", flush=True)
            start = time.perf_counter()
            run = cli.main(overrides(params, seed, label))
            (run / "train_time.json").write_text(json.dumps({"seconds": time.perf_counter() - start, "device": DEVICE}))
            print(f"  {(time.perf_counter() - start) / 60:.1f} min -> {run}", flush=True)
else:
    print("RUN_FINAL is False: no retraining. Finalists would be:", list(finalists))

In [ ]:
rows = []
for label, params in finalists.items():
    for seed, run in finished_runs(label).items():
        cache = run / "valid_hpo.json"
        if cache.is_file():
            value = json.loads(cache.read_text())["log_likelihood"]
        else:
            model = load_model(run, device=DEVICE, weights="weights_ema")
            value = float(log_likelihood(model, valid_poses, OBJECTIVE_STEPS).mean())
            cache.write_text(json.dumps({"log_likelihood": value, "steps": OBJECTIVE_STEPS, "poses": len(valid_poses)}))
        rows.append(dict(config=label, seed=seed, log_likelihood=value, run=str(run)))
if rows:
    final = pd.DataFrame(rows)
    summary = final.groupby("config")["log_likelihood"].agg(["mean", "std", "count"]).sort_values("mean", ascending=False)
    print(summary.to_string(float_format=lambda x: f"{x:.3f}"))
    winner = summary.index[0]
    print(f'\nbest: {winner}; for rsgm_huproso_comparison.ipynb: "RSGM tuned": dict(run="{FINAL_ROOT / winner}", weights="weights_ema")')
    (STUDY_DIR / "final.json").write_text(json.dumps({"finalists": finalists, "runs": rows}, indent=2))
else:
    print("no finished final runs")